# Vectorised 3-Way Parameter Grid Search

Same sweep as `2d_parameter_grid_search.ipynb` but using the
vectorised runner `run_model_core_2d_vec` which replaces per-memory
Python loops with batched tensor operations.

**Sections:**
1. Setup (same as original)
2. **Validation** — verify bit-identical outputs vs. original runner
3. **Timing** — benchmark original vs. vectorised
4. Full grid search with vectorised runner
5. Heatmap visualisations

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import time
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.model.analytic_gmm_2d import AnalyticGMM2D, make_default_gmm
from src.model.score_adapter_2d import ScoreAdapter2D
from utls.sandbox_2d_data import make_2d_grid_stimuli, compute_geometry_descriptors
from utls.runners_2d import run_model_core_2d, run_model_core_2d_vec
from utls.toy_experiments import make_high_diversity_sequences
from utls.roc_utils import roc_from_arrays, auroc_to_dprime
from utls.analysis_helpers import bootstrap_dprime_ci

SAVE_DIR = '../reports/figures/2d_grid_search'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Ready.')

## Setup: GMM prior, stimuli, sequences

In [ ]:
gmm = make_default_gmm()
X0, name_to_idx, stimulus_pool = make_2d_grid_stimuli()

# Unit-normalized score adapter
adapter = ScoreAdapter2D(gmm, normalize=True)

# Experiment sequences
ISI_VALUES = (0, 2, 16)
N_SEQUENCES = 10
SEQ_LENGTH = 81
MIN_PAIRS_PER_ISI = 5
N_MC = 10
SEED = 42
METRIC = 'cosine'

experiment_list, isi_keys = make_high_diversity_sequences(
    stimulus_pool=stimulus_pool,
    isi_values=list(ISI_VALUES),
    n_sequences=N_SEQUENCES,
    length=SEQ_LENGTH,
    min_pairs_per_isi=MIN_PAIRS_PER_ISI,
    seed=SEED,
)

print(f'{len(experiment_list)} sequences, length {len(experiment_list[0])}')
print(f'ISI values: {ISI_VALUES}')

## Validation: bit-identical outputs

Run both `run_model_core_2d` and `run_model_core_2d_vec` with
identical parameters and seeds, then assert the hit/FA score arrays
are exactly equal.

In [ ]:
test_params = [
    dict(sigma0=0.0, sigma=0.1, drift_step_size=0.0),
    dict(sigma0=0.5, sigma=0.0, drift_step_size=0.0),
    dict(sigma0=0.5, sigma=0.1, drift_step_size=0.01),
    dict(sigma0=1.0, sigma=0.2, drift_step_size=0.1),
    dict(sigma0=1.5, sigma=0.3, drift_step_size=0.05),
]

for i, params in enumerate(test_params):
    common = dict(
        X0=X0, name_to_idx=name_to_idx,
        experiment_list=experiment_list,
        score_model=adapter,
        metric=METRIC, seed=SEED,
    )
    out_orig = run_model_core_2d(**params, **common)
    out_vec  = run_model_core_2d_vec(**params, **common)

    hits_match = np.array_equal(out_orig['hits'], out_vec['hits'])
    fas_match  = np.array_equal(out_orig['fas'],  out_vec['fas'])

    status = 'PASS' if (hits_match and fas_match) else 'FAIL'
    print(f'  Config {i} (s0={params["sigma0"]}, s={params["sigma"]}, '
          f'eta={params["drift_step_size"]}): {status}'
          f'  hits={len(out_orig["hits"])}, fas={len(out_orig["fas"])}')
    if not hits_match:
        diff = np.abs(out_orig['hits'] - out_vec['hits'])
        print(f'    hits max diff: {diff.max():.2e}')
    if not fas_match:
        diff = np.abs(out_orig['fas'] - out_vec['fas'])
        print(f'    fas max diff: {diff.max():.2e}')

print('\nValidation complete.')

## Timing: original vs. vectorised

Benchmark over multiple calls to quantify the speedup.

In [ ]:
bench_params = dict(
    sigma0=0.5, sigma=0.1,
    X0=X0, name_to_idx=name_to_idx,
    experiment_list=experiment_list,
    score_model=adapter,
    drift_step_size=0.05,
    metric=METRIC, seed=SEED,
)

N_BENCH = 50

# Original
t0 = time.perf_counter()
for _ in range(N_BENCH):
    run_model_core_2d(**bench_params)
t_orig = (time.perf_counter() - t0) / N_BENCH

# Vectorised
t0 = time.perf_counter()
for _ in range(N_BENCH):
    run_model_core_2d_vec(**bench_params)
t_vec = (time.perf_counter() - t0) / N_BENCH

print(f'Original:   {t_orig*1000:.1f} ms/call')
print(f'Vectorised: {t_vec*1000:.1f} ms/call')
print(f'Speedup:    {t_orig/t_vec:.2f}x')

## Define grid + MC sweep function (vectorised)

In [ ]:
# --- Fine-grained parameter grids ---
sigma0_grid = np.array([0.0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0])
sigma_grid  = np.array([0.0, 0.025, 0.05, 0.1, 0.15, 0.2, 0.3])
eta_grid    = np.array([0.0, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2])

print(f'Grid size: {len(sigma0_grid)} x {len(sigma_grid)} x {len(eta_grid)} = '
      f'{len(sigma0_grid) * len(sigma_grid) * len(eta_grid)} configurations')
print(f'With {N_MC} MC reps each and {len(ISI_VALUES)} ISIs')


def run_mc_dprime(sigma0, sigma, eta, isi_values=ISI_VALUES, n_mc=N_MC, seed=SEED):
    """Run MC sweep using vectorised runner and return d' per ISI."""
    runner_isi_values = [isi + 1 for isi in isi_values]
    score_type = 'distance'

    all_isi_hits = defaultdict(list)
    all_fas = []

    for rep in range(n_mc):
        run = run_model_core_2d_vec(
            sigma0=sigma0, sigma=sigma,
            X0=X0, name_to_idx=name_to_idx,
            experiment_list=experiment_list,
            score_model=adapter,
            drift_step_size=eta,
            metric=METRIC,
            seed=seed * 10_000 + rep,
        )
        for risi in runner_isi_values:
            all_isi_hits[risi].extend(run['isi_hit_dists'].get(risi, []))
        all_fas.extend(run['fas'])

    all_fas = np.array(all_fas, dtype=float)
    result = {}

    for exp_isi, risi in zip(isi_values, runner_isi_values):
        hits_raw = all_isi_hits.get(risi, [])
        if len(hits_raw) < 3:
            result[exp_isi] = np.nan
            continue
        hits = np.array([s for s, t in hits_raw], dtype=float)
        roc = roc_from_arrays(hits, all_fas, score_type=score_type)
        if roc is None:
            result[exp_isi] = np.nan
        else:
            _, _, auc_val = roc
            result[exp_isi] = auroc_to_dprime(auc_val)

    return result

## Run the full grid search (vectorised)

In [ ]:
# Storage: results[isi][(i_s0, i_sig, i_eta)] = d'
results = {isi: np.full((len(sigma0_grid), len(sigma_grid), len(eta_grid)), np.nan)
           for isi in ISI_VALUES}

total = len(sigma0_grid) * len(sigma_grid) * len(eta_grid)
count = 0
t_start = time.perf_counter()

for i_s0, s0 in enumerate(sigma0_grid):
    for i_sig, sig in enumerate(sigma_grid):
        for i_eta, eta in enumerate(eta_grid):
            dp = run_mc_dprime(s0, sig, eta)
            for isi in ISI_VALUES:
                results[isi][i_s0, i_sig, i_eta] = dp.get(isi, np.nan)
            count += 1
            if count % 50 == 0:
                elapsed = time.perf_counter() - t_start
                rate = count / elapsed
                remaining = (total - count) / rate
                print(f'  {count}/{total} done  ({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining)')

t_total = time.perf_counter() - t_start
print(f'Grid search complete: {count} configurations in {t_total:.1f}s ({t_total/60:.1f} min).')

## Heatmaps: d' as f(sigma, eta) for each ISI, sliced by sigma0

Each row = one ISI, each column = one sigma0 level.
Color = d' value.

In [ ]:
# Select a subset of sigma0 levels for columns
s0_slice_indices = [0, 2, 4, 6]  # 0.0, 0.25, 0.75, 1.5
s0_slice_vals = sigma0_grid[s0_slice_indices]

fig, axes = plt.subplots(len(ISI_VALUES), len(s0_slice_indices),
                         figsize=(4 * len(s0_slice_indices), 3.5 * len(ISI_VALUES)),
                         sharex=True, sharey=True)

vmin = 0
vmax = np.nanmax([results[isi] for isi in ISI_VALUES])
vmax = min(vmax, 5.0)  # cap for visual clarity

for row, isi in enumerate(ISI_VALUES):
    for col, i_s0 in enumerate(s0_slice_indices):
        ax = axes[row, col]
        data = results[isi][i_s0, :, :]  # [n_sigma, n_eta]
        im = ax.imshow(data, aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax, cmap='viridis',
                       extent=[eta_grid[0], eta_grid[-1],
                               sigma_grid[0], sigma_grid[-1]])
        if row == 0:
            ax.set_title(f'\u03c3\u2080={sigma0_grid[i_s0]:.2f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={isi}\n\u03c3 (diffusive)', fontsize=10)
        if row == len(ISI_VALUES) - 1:
            ax.set_xlabel('\u03b7 (drift step)', fontsize=10)

fig.suptitle("d' across parameter space (rows=ISI, cols=\u03c3\u2080)", fontsize=13, y=1.01)
fig.colorbar(im, ax=axes, shrink=0.6, label="d'")
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/dprime_heatmaps_sigma_eta_vec.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmaps: d' as f(sigma0, eta) for each ISI, sliced by sigma

Complementary view: fix diffusive noise, sweep encoding noise vs drift.

In [ ]:
sig_slice_indices = [0, 2, 4, 6]  # 0.0, 0.05, 0.15, 0.3
sig_slice_vals = sigma_grid[sig_slice_indices]

fig, axes = plt.subplots(len(ISI_VALUES), len(sig_slice_indices),
                         figsize=(4 * len(sig_slice_indices), 3.5 * len(ISI_VALUES)),
                         sharex=True, sharey=True)

for row, isi in enumerate(ISI_VALUES):
    for col, i_sig in enumerate(sig_slice_indices):
        ax = axes[row, col]
        data = results[isi][:, i_sig, :]  # [n_sigma0, n_eta]
        im = ax.imshow(data, aspect='auto', origin='lower',
                       vmin=vmin, vmax=vmax, cmap='viridis',
                       extent=[eta_grid[0], eta_grid[-1],
                               sigma0_grid[0], sigma0_grid[-1]])
        if row == 0:
            ax.set_title(f'\u03c3={sigma_grid[i_sig]:.3f}', fontsize=11)
        if col == 0:
            ax.set_ylabel(f'ISI={isi}\n\u03c3\u2080 (encoding)', fontsize=10)
        if row == len(ISI_VALUES) - 1:
            ax.set_xlabel('\u03b7 (drift step)', fontsize=10)

fig.suptitle("d' across parameter space (rows=ISI, cols=\u03c3)", fontsize=13, y=1.01)
fig.colorbar(im, ax=axes, shrink=0.6, label="d'")
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/dprime_heatmaps_sigma0_eta_vec.png', dpi=150, bbox_inches='tight')
plt.show()

## ISI decay summary: d'(ISI=0) - d'(ISI=16) as f(sigma0, eta) at fixed sigma

How much does memory degrade over time? Large positive values = strong ISI decay.

In [ ]:
# Compute ISI decay: d'(ISI=0) - d'(ISI=16)
decay = results[0] - results[16]  # [n_sigma0, n_sigma, n_eta]

fig, axes = plt.subplots(1, len(sig_slice_indices),
                         figsize=(4 * len(sig_slice_indices), 4),
                         sharex=True, sharey=True)

dmax = np.nanmax(np.abs(decay))
dmax = min(dmax, 3.0)

for col, i_sig in enumerate(sig_slice_indices):
    ax = axes[col]
    data = decay[:, i_sig, :]  # [n_sigma0, n_eta]
    im = ax.imshow(data, aspect='auto', origin='lower',
                   vmin=-dmax, vmax=dmax, cmap='RdBu_r',
                   extent=[eta_grid[0], eta_grid[-1],
                           sigma0_grid[0], sigma0_grid[-1]])
    ax.set_title(f'\u03c3={sigma_grid[i_sig]:.3f}', fontsize=11)
    ax.set_xlabel('\u03b7 (drift step)', fontsize=10)
    if col == 0:
        ax.set_ylabel('\u03c3\u2080 (encoding)', fontsize=10)

fig.suptitle("ISI decay: d'(ISI=0) \u2212 d'(ISI=16)", fontsize=13, y=1.02)
fig.colorbar(im, ax=axes, shrink=0.8, label="\u0394d'")
fig.tight_layout()
fig.savefig(f'{SAVE_DIR}/isi_decay_heatmaps_vec.png', dpi=150, bbox_inches='tight')
plt.show()

## Save grid results for reuse

In [ ]:
np.savez(f'{SAVE_DIR}/grid_search_results_vec.npz',
         sigma0_grid=sigma0_grid,
         sigma_grid=sigma_grid,
         eta_grid=eta_grid,
         isi_values=np.array(ISI_VALUES),
         **{f'dprime_isi{isi}': results[isi] for isi in ISI_VALUES})
print(f'Saved to {SAVE_DIR}/grid_search_results_vec.npz')